In [ ]:
import pandas as pd
import pandas as pd
from sqlalchemy import create_engine, text

# 数据库配置
username = "XXXXXX"
password = "YYYYYY"
host = "localhost"
port = 5432
database = "eyewear-data"

# 创建连接
engine = create_engine(
    f"postgresql+psycopg2://{username}:{password}@{host}:{port}/{database}"
)


sql_1 = """
SELECT * FROM "CustomerInfo"
"""
df = pd.read_sql(sql_1, engine)
df

,customer_id,customer_m3_code,customer_name,gender,age,birth_date,region,country,city,registration_date,last_purchase_date,total_orders,total_spend,customer_type,customer_hierarchy,channel_source,preferred_store_id,is_active
0,6201,CUST-16200,Joshua Malone,Male,56,1969-11-01,Florida,United States,Other,2023-11-13,2024-06-16,4,1584.68,VIP Loyal Customers,3,Offline,12.0,True
1,6203,CUST-16202,Megan Green,Female,59,1966-08-06,Texas,United States,Other,2024-05-02,2024-04-03,1,247.69,One-time Customers,1,Online,89.0,True
2,6207,CUST-16206,Nicole Peterson,Male,37,1988-07-30,California,United States,Northridge,2018-05-14,2024-02-29,2,784.87,Lens Customers,2,Offline,6.0,True
3,6208,CUST-16207,Eric Baldwin,Female,53,1973-07-10,California,United States,Santa Clara,2021-07-29,2023-09-04,5,1666.43,Regular Customers,4,Offline,3.0,True
4,6210,CUST-16209,Pedro White,Female,26,1999-11-20,Texas,United States,Other,2021-01-25,2024-11-01,2,832.02,Lens Customers,2,Offline,62.0,True
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
159995,6189,CUST-16188,Robert Garrison,Female,57,1969-01-07,Washington,United States,Other,2024-08-30,2023-10-29,1,197.93,One-time Customers,1,Offline,38.0,True
159996,6194,CUST-16193,Lance Stafford,Female,47,1978-08-01,Illinois,United States,Other,2016-10-31,2023-09-26,4,1421.85,Regular Customers,4,Offline,21.0,True
159997,6200,CUST-16199,David Holloway,Male,53,1972-10-15,Florida,United States,Miami,2022-05-30,2024-11-19,2,531.11,Lens Customers,2,Offline,51.0,True
159998,4060,CUST-14059,Carolyn Anderson,Male,27,1998-09-10,Colorado,United States,Other,2020-03-13,None,0,0.00,None,None,None,NaN,True


In [5]:
df.columns

Index(['customer_id', 'customer_m3_code', 'customer_name', 'gender', 'age',
       'birth_date', 'region', 'country', 'city', 'registration_date',
       'last_purchase_date', 'total_orders', 'total_spend', 'customer_type',
       'customer_hierarchy', 'channel_source', 'preferred_store_id',
       'is_active'],
      dtype='object')

### B1. 客户聚类结果

In [ ]:
sql_2 = """
WITH rfm_base AS (
SELECT
ci.customer_id,
ci.customer_type,
ci.customer_hierarchy,
DATE_PART('day', '2025-06-01' - MAX(o.order_date))::INT AS recency,
COUNT(DISTINCT o.order_id) AS frequency,
SUM(CASE WHEN oi.is_free_gift = FALSE OR oi.is_free_gift IS NULL THEN oi.line_price_before_tax ELSE 0 END) AS monetary
FROM "CustomerInfo" ci
LEFT JOIN "Order" o
ON ci.customer_id = o.customer_id AND o.order_status IN ('Completed','Shipped')
LEFT JOIN "OrderItem" oi
ON o.order_id = oi.order_id
GROUP BY ci.customer_id, ci.customer_type, ci.customer_hierarchy
),
rfm_scores AS (
SELECT
customer_id,
customer_type,
customer_hierarchy,
recency,
frequency,
monetary,
-- recency: 较小越好，先按 recency 升序分箱，给 5 表示最近期
NTILE(5) OVER (ORDER BY recency ASC) AS r_bin,
-- frequency: 越大越好
NTILE(5) OVER (ORDER BY frequency DESC NULLS LAST) AS f_bin,
-- monetary: 越大越好
NTILE(5) OVER (ORDER BY monetary DESC NULLS LAST) AS m_bin
FROM rfm_base
),
rfm_final AS (
SELECT
customer_id,
customer_type,
customer_hierarchy,
recency,
frequency,
monetary,
r_bin AS r_score,
f_bin AS f_score,
m_bin AS m_score,
-- 合并成 segment 字符串，比如 5-4-2
(r_bin::text || '-' || f_bin::text || '-' || m_bin::text) AS rfm_segment,
-- 简单生命周期规则（可调整）：
-- New: registration 在 30 天内且 frequency=1
-- Active: recency <= 90 且 f_score>=3
-- At Risk: recency > 90 且 recency <= 365
-- Churned: recency > 365 或 frequency=0
CASE
WHEN recency IS NULL THEN 'NoOrders'
WHEN recency <= 30 AND frequency <= 1 THEN 'New'
WHEN recency <= 90 AND f_bin >= 3 THEN 'Active'
WHEN recency > 90 AND recency <= 365 THEN 'At Risk'
WHEN recency > 365 OR frequency = 0 THEN 'Churned'
ELSE 'At Risk'
END AS lifecycle_stage,
-- 简单流失风险标记（Churned 或 At Risk 视为高风险）
CASE
WHEN (recency > 365 OR frequency = 0) THEN 1
WHEN (recency > 90 AND recency <= 365) THEN 1
ELSE 0
END AS churn_risk
FROM rfm_scores
)
SELECT * FROM rfm_final
ORDER BY customer_id;
"""


df_rfm = pd.read_sql(sql_2, engine)
df_rfm.to_parquet("2023-2024_customer_cluster_rfm.parquet", engine='fastparquet', index=False)
print("数据已保存为2023-2024_customer_cluster_rfm.parquet")
df_rfm


数据已保存为2023-2024_customer_cluster_rfm.parquet


,customer_id,customer_type,customer_hierarchy,recency,frequency,monetary,r_score,f_score,m_score,rfm_segment,lifecycle_stage,churn_risk
0,1,Promotional Sensitive Customers,0,570.0,2,531.01,4,4,4,4-4-4,Churned,1
1,2,Regular Customers,4,630.0,2,747.58,5,3,4,5-3-4,Churned,1
2,3,One-time Customers,1,680.0,1,396.67,5,5,4,5-5-4,Churned,1
3,4,Regular Customers,4,405.0,2,384.56,4,4,5,4-4-5,Churned,1
4,5,Promotional Sensitive Customers,0,391.0,1,346.47,3,5,5,3-5-5,Churned,1
...,...,...,...,...,...,...,...,...,...,...,...,...
159995,159996,One-time Customers,1,164.0,1,251.76,1,5,5,1-5-5,At Risk,1
159996,159997,VIP Loyal Customers,3,172.0,6,1998.35,1,1,1,1-1-1,At Risk,1
159997,159998,None,None,NaN,0,NaN,5,5,5,5-5-5,NoOrders,1
159998,159999,Regular Customers,4,342.0,2,1349.33,3,4,2,3-4-2,At Risk,1


### agg_cluster_monthly —— 各 cluster 月度销售

In [14]:
sql_cluster_monthly_sale = """
SELECT
ci.customer_hierarchy,
ci.customer_type,
EXTRACT(YEAR FROM o.order_date)::INT AS year,
EXTRACT(MONTH FROM o.order_date)::INT AS month,
TO_CHAR(o.order_date, 'YYYY-MM') AS year_month,

SUM(
CASE WHEN oi.is_free_gift = FALSE OR oi.is_free_gift IS NULL
THEN oi.line_price_before_tax ELSE 0 END
) AS total_sales_amount,

SUM(
CASE WHEN oi.is_free_gift = FALSE OR oi.is_free_gift IS NULL
THEN oi.quantity ELSE 0 END
) AS total_sales_qty,

COUNT(DISTINCT o.order_id) AS order_count,

COUNT(DISTINCT ci.customer_id) AS customer_count,

CASE WHEN COUNT(DISTINCT o.order_id)=0 THEN NULL
ELSE ROUND(
SUM(
CASE WHEN oi.is_free_gift = FALSE OR oi.is_free_gift IS NULL
THEN oi.line_price_before_tax ELSE 0 END
)::NUMERIC
/ COUNT(DISTINCT o.order_id)::NUMERIC, 2)
END AS aov,

SUM(
CASE WHEN oi.is_free_gift = FALSE OR oi.is_free_gift IS NULL
THEN oi.line_price_before_tax - (COALESCE(pi.cost_price,0)*oi.quantity) ELSE 0 END
) AS gross_profit,

CASE WHEN SUM(
CASE WHEN oi.is_free_gift = FALSE OR oi.is_free_gift IS NULL
THEN oi.line_price_before_tax ELSE 0 END) = 0
THEN NULL
ELSE ROUND(
SUM(
CASE WHEN oi.is_free_gift = FALSE OR oi.is_free_gift IS NULL
THEN oi.line_price_before_tax - (COALESCE(pi.cost_price,0)*oi.quantity) ELSE 0 END
)::NUMERIC
/
SUM(
CASE WHEN oi.is_free_gift = FALSE OR oi.is_free_gift IS NULL
THEN oi.line_price_before_tax ELSE 0 END
)::NUMERIC, 4)
END AS gross_margin_rate

FROM "CustomerInfo" ci 
LEFT JOIN "Order" o
ON ci.customer_id = o.customer_id AND o.order_status IN ('Completed','Shipped')
LEFT JOIN "OrderItem" oi
ON o.order_id = oi.order_id
LEFT JOIN "ProductInfo" pi
ON oi.product_id = pi.product_id

WHERE ci.customer_hierarchy IS NOT NULL

GROUP BY
ci.customer_hierarchy,
ci.customer_type,
year, month, year_month

ORDER BY
ci.customer_hierarchy, year, month;
"""


df_cluster_monthly_sale = pd.read_sql(sql_cluster_monthly_sale, engine)
df_cluster_monthly_sale.to_parquet("2023-2024_customer_cluster_monthly_sale.parquet", engine='fastparquet', index=False)
print("数据已保存为2023-2024_customer_cluster_monthly_sale.parquet")
df_cluster_monthly_sale

数据已保存为2023-2024_customer_cluster_monthly_sale.parquet


,customer_hierarchy,customer_type,year,month,year_month,total_sales_amount,total_sales_qty,order_count,customer_count,aov,gross_profit,gross_margin_rate
0,0,Promotional Sensitive Customers,2023,1,2023-01,607999.21,2839,2001,1949,303.85,294585.29,0.4845
1,0,Promotional Sensitive Customers,2023,2,2023-02,534585.98,2494,1728,1686,309.37,261354.90,0.4889
2,0,Promotional Sensitive Customers,2023,3,2023-03,518262.72,2418,1732,1695,299.23,253858.45,0.4898
3,0,Promotional Sensitive Customers,2023,4,2023-04,517028.72,2390,1719,1681,300.77,255010.84,0.4932
4,0,Promotional Sensitive Customers,2023,5,2023-05,550677.31,2536,1836,1790,299.93,272887.12,0.4955
...,...,...,...,...,...,...,...,...,...,...,...,...
115,4,Regular Customers,2024,8,2024-08,2226010.41,8649,5673,5483,392.39,1222401.03,0.5491
116,4,Regular Customers,2024,9,2024-09,1997658.13,7723,5047,4886,395.81,1098233.97,0.5498
117,4,Regular Customers,2024,10,2024-10,2868663.53,11181,7513,7158,381.83,1550007.78,0.5403
118,4,Regular Customers,2024,11,2024-11,2797499.10,10914,7313,6967,382.54,1514445.73,0.5414
